In [2]:
import json
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from tqdm import tqdm

# ===============================
# CONFIG
# ===============================
INPUT_FILE = "/kaggle/input/qa-data1/qa_data.json"
OUTPUT_FILE = "/kaggle/working/qa_data_summarized_bart.json"

MODEL_NAME = "facebook/bart-large-cnn"
MAX_INPUT_LEN = 1024
MAX_SUMMARY_LEN = 200

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ===============================
# LOAD MODEL
# ===============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(DEVICE)
model.eval()

# ===============================
# QUESTION-AWARE SUMMARIZATION
# ===============================
def summarize_knowledge_with_question(knowledge, question):
    prompt = (
        "Summarize the following knowledge to answer the question.\n\n"
        f"Question: {question}\n\n"
        f"Knowledge: {knowledge}"
    )

    inputs = tokenizer(
        prompt,
        truncation=True,
        max_length=MAX_INPUT_LEN,
        return_tensors="pt"
    ).to(DEVICE)

    with torch.no_grad():
        summary_ids = model.generate(
            **inputs,
            max_length=MAX_SUMMARY_LEN,
            min_length=50,
            num_beams=4,
            length_penalty=2.0
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)

# ===============================
# LOAD DATA (JSON / JSONL)
# ===============================
def load_json_or_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        first_char = f.read(1)
        f.seek(0)
        if first_char == "[":
            data = json.load(f)
        else:
            for line in f:
                line = line.strip()
                if line:
                    data.append(json.loads(line))
    return data

data = load_json_or_jsonl(INPUT_FILE)
print(f"Loaded {len(data)} samples")

# ===============================
# PROCESS + SAVE
# ===============================
new_data = []

for sample in tqdm(data, desc="Summarizing with BART"):
    summarized_knowledge = summarize_knowledge_with_question(
        sample["knowledge"],
        sample["question"]
    )

    new_data.append({
        "knowledge": summarized_knowledge,
        "question": sample["question"],
        "right_answer": sample["right_answer"],
        "hallucinated_answer": sample["hallucinated_answer"]
    })

# ===============================
# SAVE OUTPUT
# ===============================
with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    json.dump(new_data, f, indent=2, ensure_ascii=False)

print(f"\n✅ File saved at: {OUTPUT_FILE}")


Loaded 10000 samples


Summarizing with BART: 100%|██████████| 10000/10000 [2:22:27<00:00,  1.17it/s] 


✅ File saved at: /kaggle/working/qa_data_summarized_bart.json
